# 19. Agent Loop From Scratch (In-Depth)

**Tier:** Agent Engineering
**Estimated time:** 60 minutes
**Prerequisites:** 18
**Source material:** @sairahul1 ; @theahmadosman build-first roadmap (https://x.com/theahmadosman/status/2062343535144436073)

## What You'll Learn
- The complete agent loop in raw Anthropic SDK — no framework, ~100 lines, every step printed
- How to register multiple tools and let the model choose between them
- How to enforce a budget so an agent can't loop forever

## Why This Matters
Frameworks (notebook 20's LangGraph) hide this loop behind nice abstractions. You should build it once by hand so that when a framework misbehaves, you know exactly what it's doing underneath. This is the single most important notebook in the tier for understanding *how agents actually work*.

> **Two versions:** this is the in-depth build. Notebook **19b** is a ~30-line minimal version of the same loop — reach for 19 when you need to understand or debug the mechanism, and 19b when you just need a working agent fast.


## The loop, in plain English

Notebook 18 ran the tool-call round-trip once. An agent is that round-trip in a `while` loop:

```
messages = [the user's goal]
loop:
    response = model(messages, tools)
    if response.stop_reason != "tool_use":
        return response   # the model is done
    for each tool the model requested:
        result = run_that_tool(its_inputs)
        record the result
    append the model's request AND our results to messages
    # loop again — the model now sees what the tools returned
```

That's the whole thing. The only subtlety is **bookkeeping**: each turn you must append *both* the assistant's message (containing its `tool_use` blocks) *and* a user message containing a matching `tool_result` for every `tool_use` — keyed by `tool_use_id`. Get one ID wrong and the API rejects the turn. We'll print the full trace so you can see every message accumulate.

Our agent gets three tools, mirroring a real assistant: `web_search` (mocked so the notebook runs offline), `read_file`, and `write_file` (scoped to a temp directory so it's safe).


In [ ]:
import os, json, pathlib, tempfile
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"   # production default: claude-opus-4-8

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print(f"Anthropic ready — agent will call {TEACH_MODEL}.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — the loop will print a skip message. The code is still readable.")

# A scratch directory so write_file/read_file can't touch anything important.
WORKDIR = pathlib.Path(tempfile.mkdtemp(prefix="agent19_"))
print("Agent scratch dir:", WORKDIR)


## Step 1: define the tools (schema + implementation)

Each tool is a pair: a JSON schema the model reads, and a Python function we run. We keep a registry mapping tool name → function.


In [ ]:
# A tiny fake "search index" so web_search returns deterministic results offline.
_FAKE_WEB = {
    "helios x1 warranty": "The Helios X1 has a 2-year limited warranty on manufacturing defects.",
    "helios x1 reset": "To reset the Helios X1 wifi: hold the reset button 10s while powered on.",
    "python list comprehension": "A concise way to build lists: [f(x) for x in xs if cond(x)].",
}

def web_search(query):
    q = query.lower().strip()
    for key, val in _FAKE_WEB.items():
        if key in q or q in key:
            return val
    return "No results found (this is a mock search index)."

def read_file(filename):
    path = WORKDIR / filename
    if not path.exists():
        return f"error: {filename} does not exist"
    return path.read_text()

def write_file(filename, content):
    path = WORKDIR / filename
    path.write_text(content)
    return f"wrote {len(content)} chars to {filename}"

TOOL_IMPLEMENTATIONS = {"web_search": web_search, "read_file": read_file, "write_file": write_file}

TOOL_SCHEMAS = [
    {"name": "web_search", "description": "Search the web for a fact. Returns a short snippet.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "read_file", "description": "Read a text file from the agent's working directory.",
     "input_schema": {"type": "object", "properties": {"filename": {"type": "string"}}, "required": ["filename"]}},
    {"name": "write_file", "description": "Write text to a file in the agent's working directory.",
     "input_schema": {"type": "object", "properties": {
         "filename": {"type": "string"}, "content": {"type": "string"}}, "required": ["filename", "content"]}},
]
print("Registered tools:", list(TOOL_IMPLEMENTATIONS))


## Step 2: the loop itself

About 30 lines of real logic. Everything else in this notebook is tools and printing. Read the comments top to bottom — this *is* the agent.


In [ ]:
def run_agent(goal, max_steps=8, verbose=True):
    """Raw Anthropic agent loop. Returns the final text answer."""
    if not HAS_ANTHROPIC:
        print("  [skipped: no ANTHROPIC_API_KEY]")
        return None

    messages = [{"role": "user", "content": goal}]
    for step in range(1, max_steps + 1):
        # 1. Ask the model what to do next, given everything so far.
        resp = client.messages.create(
            model=TEACH_MODEL, max_tokens=600,
            system="You are a capable assistant. Use tools when they help; otherwise answer directly.",
            tools=TOOL_SCHEMAS, messages=messages,
        )
        if verbose:
            print(f"\n=== step {step} | stop_reason={resp.stop_reason} ===")

        # 2. If the model is done, return its text.
        if resp.stop_reason != "tool_use":
            answer = "".join(b.text for b in resp.content if b.type == "text")
            if verbose:
                print("FINAL:", answer)
            return answer

        # 3. Otherwise run every tool it requested and collect the results.
        messages.append({"role": "assistant", "content": resp.content})   # remember its request
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":
                impl = TOOL_IMPLEMENTATIONS[block.name]
                result = impl(**block.input)
                if verbose:
                    print(f"  tool: {block.name}({block.input}) -> {result[:80]}")
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})

        # 4. Feed the results back and loop.
        messages.append({"role": "user", "content": tool_results})

    return "stopped: hit max_steps without finishing"


## Step 3: run it on a multi-tool task

This goal forces the model to chain tools: search for a fact, then write it to a file, then read it back to confirm. Watch the trace — the model decides the order.


In [ ]:
answer = run_agent(
    "Find the Helios X1 warranty length using web_search, save it to a file called warranty.txt, "
    "then read the file back to confirm what was saved. Report what the file contains."
)


## Visualizing the trace: tools used per step

We instrument a run to count how many tool calls happened at each step, so you can see the agent's "shape."


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

def run_agent_instrumented(goal, max_steps=8):
    if not HAS_ANTHROPIC:
        return []
    messages = [{"role": "user", "content": goal}]
    tools_per_step = []
    for step in range(max_steps):
        resp = client.messages.create(model=TEACH_MODEL, max_tokens=600,
            system="Use tools when helpful.", tools=TOOL_SCHEMAS, messages=messages)
        if resp.stop_reason != "tool_use":
            tools_per_step.append(0)
            break
        messages.append({"role": "assistant", "content": resp.content})
        results, n = [], 0
        for b in resp.content:
            if b.type == "tool_use":
                n += 1
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": TOOL_IMPLEMENTATIONS[b.name](**b.input)})
        tools_per_step.append(n)
        messages.append({"role": "user", "content": results})
    return tools_per_step

counts = run_agent_instrumented(
    "Search for 'helios x1 reset', write the answer to reset.txt, then read it back.")
if counts:
    plt.figure(figsize=(7, 3.5))
    plt.bar(range(1, len(counts) + 1), counts, color="#4C72B0")
    plt.title("Tool calls per agent step"); plt.xlabel("Step"); plt.ylabel("# tool calls")
    plt.yticks(range(0, max(counts) + 2)); plt.tight_layout(); plt.show()
else:
    print("  [no trace — API unavailable]")


*The last step is usually 0 tool calls — that's the agent deciding it has everything it needs and writing the final answer instead of calling another tool.*


## Exercises


In [ ]:
# Exercise 1 (Warm-up): Add a fact to the mock index
# Task: Add an entry to _FAKE_WEB (e.g. "helios x1 battery": "..."), then give run_agent a goal
#       that requires searching for it. Confirm the agent retrieves your new fact.
# Hint: The agent can only "find" what web_search can return — the mock index IS its world here.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): A list_files tool
# Task: Add a fourth tool, list_files(), that returns the names of files in WORKDIR. Register its
#       schema and implementation, then ask the agent to write two files and list them.
# Hint: Follow the exact pattern of read_file — a schema dict in TOOL_SCHEMAS plus an entry in
#       TOOL_IMPLEMENTATIONS. No change to run_agent is needed.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): Budget enforcement
# Task: Add a `max_tool_calls` budget to run_agent. Count total tool calls across all steps; once
#       the budget is exceeded, stop looping and ask the model to SUMMARIZE what it found so far
#       and stop (instead of calling more tools).
# Hint: When over budget, make the final model call WITHOUT passing tools= — that forces a text
#       answer. This is exactly the safeguard the P3 capstone relies on.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
_FAKE_WEB["helios x1 battery"] = "The Helios X1 battery lasts about 8 hours per charge."
run_agent("Use web_search to find how long the Helios X1 battery lasts, then state it.")

# Exercise 2
def list_files(): return ", ".join(p.name for p in WORKDIR.iterdir()) or "(empty)"
TOOL_IMPLEMENTATIONS["list_files"] = lambda **kw: list_files()
TOOL_SCHEMAS.append({"name": "list_files", "description": "List files in the working directory.",
                     "input_schema": {"type": "object", "properties": {}}})
run_agent("Write 'a' to one.txt and 'b' to two.txt, then list the files.")

# Exercise 3
def run_agent_budgeted(goal, max_steps=8, max_tool_calls=3):
    messages = [{"role": "user", "content": goal}]
    used = 0
    for step in range(max_steps):
        over = used >= max_tool_calls
        kwargs = dict(model=TEACH_MODEL, max_tokens=500, messages=messages,
                      system="Use tools when helpful.")
        if not over:
            kwargs["tools"] = TOOL_SCHEMAS          # withholding tools forces a text summary
        resp = client.messages.create(**kwargs)
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                used += 1
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": TOOL_IMPLEMENTATIONS[b.name](**b.input)})
        messages.append({"role": "user", "content": results})
    return "stopped"
print(run_agent_budgeted("Search helios x1 warranty, reset, and battery, then summarize.", max_tool_calls=2))
```
</details>


## Key Takeaways
- The agent loop is ~30 lines: call model → if it wants a tool, run it and append the result → repeat until `stop_reason != "tool_use"`.
- You must append both the assistant's `tool_use` message and a matching `tool_result` (by `tool_use_id`) every turn — that bookkeeping is the only fiddly part.
- Tools are a registry: a schema the model reads + a function you run. Adding a tool needs no change to the loop.
- A budget (max steps or max tool calls) is essential — without it an agent can loop indefinitely. Withholding `tools=` on the final call forces a text answer.
- This raw loop is what every agent framework wraps; build it once so the frameworks hold no mystery.

## What's Next
Notebook **19b — Agent Loop (Minimal)** strips this same loop down to ~30 lines you can paste into any project when you just need a working agent fast.
